# Agentic Mechanics: Planning and Tool Use

## All-in-one notebook

This single notebook runs the whole day in order: a first tool-use loop, a planning
pattern (ReAct), a multi-tool agent, reflection and replanning, and a capstone. You
install once and load your key once at the top, then work straight down the notebook.

## Setup: run these first

Every notebook starts the same way: install the library, load your key, and
pick a model. Run the next three cells in order.

**Save yourself time:** store your API key once in Colab Secrets (the key icon on
the left sidebar). Add a secret named `ANTHROPIC_API_KEY`, paste your key as the
value, and turn on "Notebook access". After that, every notebook picks it up
automatically and you never paste it again.

In [ ]:
# 1) Install the Anthropic library (Colab forgets it between sessions, so we
#    install each time). The -q just keeps the output quiet.
!pip install -q anthropic

In [ ]:
# 2) Load your Anthropic API key.
#    Easiest: store it once in Colab Secrets (key icon on the left), named
#    ANTHROPIC_API_KEY, with "Notebook access" on. Then this reads it for you.
import anthropic

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = None  # not on Colab, or no secret set yet

if not api_key:
    # Fallback: ask once if no secret is found. The input is hidden and not saved.
    from getpass import getpass
    api_key = getpass("Paste your Anthropic API key (input is hidden): ")

client = anthropic.Anthropic(api_key=api_key)
print("Your client is ready.")

In [ ]:
# 3) Choose the model. claude-haiku-5-5 is fast and cheap, good for practice. For stronger
#    answers to harder questions, switch this to "claude-opus-5-5".
MODEL = "claude-haiku-5-5"
print("Using model:", MODEL)

## Part 1: Your first tool-use loop

Give the model a calculator and watch the loop: ask, run, return, continue.

In [ ]:
import ast
import operator

# A tool is described by a name, a plain description, and the inputs it needs.
# The model reads the description to decide when to use the tool.
calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression and return the result. Use this whenever the user asks for a calculation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "An arithmetic expression, for example '12 * 37'.",
            }
        },
        "required": ["expression"],
    },
}

# The safe operations our calculator allows: plus, minus, times, divide, and
# negative numbers. Nothing else is permitted.
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    # Walk the parsed expression and only allow numbers and the safe operations
    # above. Anything else (names, function calls) raises an error, so no
    # arbitrary code can run. This is the safe alternative to eval().
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")


def run_calculator(expression):
    # Beginners can treat this as a black box: give it an expression, get the
    # result. It parses the text and evaluates only safe arithmetic, so it
    # cannot run arbitrary code.
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree))
    except Exception:
        return "Error: I could only handle a simple arithmetic expression like '12 * 37'."

In [ ]:
# A single tool-use loop, by hand.
tools = [calculator_tool]
messages = [{"role": "user", "content": "What is 12 times 37? Use the calculator."}]

response = client.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)
messages.append({"role": "assistant", "content": response.content})

tool_results = []
for block in response.content:
    if block.type == "tool_use":
        print("The model asked to run", block.name, "with", block.input)
        result = run_calculator(block.input["expression"])
        print("Our code computed:", result)
        tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})

messages.append({"role": "user", "content": tool_results})
followup = client.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)
for block in followup.content:
    if block.type == "text":
        print("Model's final answer:", block.text)

## Part 2: A planning pattern (ReAct)

The agent thinks, acts, and observes, over and over. We build the loop once here and reuse it for the rest of the notebook.

In [ ]:
# Two small tools. One looks up an approximate fact, one does arithmetic.
facts = {
    "geneva population": "200000 (approximate, for the workshop)",
    "zurich population": "430000 (approximate, for the workshop)",
    "bern population": "135000 (approximate, for the workshop)",
}

lookup_tool = {
    "name": "lookup_fact",
    "description": "Look up an approximate fact by name, for example 'geneva population'. Use this when you need a number you do not know.",
    "input_schema": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "The fact to look up, lowercase."}
        },
        "required": ["name"],
    },
}


def run_lookup(name):
    return facts.get(name.lower().strip(), "Error: I do not have that fact. Try another name.")

In [ ]:
tools = [lookup_tool, calculator_tool]

# A small helper that runs whichever tool the model asked for.
def run_tool(name, tool_input):
    if name == "lookup_fact":
        return run_lookup(tool_input["name"])
    if name == "calculator":
        return run_calculator(tool_input["expression"])
    return "Error: unknown tool."

SYSTEM = (
    "You are a careful assistant. Think step by step. "
    "When you need a fact or a calculation, use the tools provided, one step at a time. "
    "After you have everything you need, give a short final answer."
)

In [ ]:
# The ReAct loop, defined once and reused for every part below.
# It reads the current tools, run_tool, and SYSTEM each time it is called.
def run_agent(question, max_steps=6):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=tools, messages=messages,
        )
        for block in response.content:
            if block.type == "text" and block.text.strip():
                print("Thought:", block.text.strip())
        if response.stop_reason != "tool_use":
            for block in response.content:
                if block.type == "text":
                    return block.text
            return ""
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print("Action:", block.name, block.input)
                observation = run_tool(block.name, block.input)
                print("Observation:", observation)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": observation})
        messages.append({"role": "user", "content": tool_results})
    return "Stopped after the maximum number of steps."

In [ ]:
print(run_agent("What is the combined population of Geneva and Zurich? Use the tools."))

## Part 3: A multi-tool agent

Add a third tool and watch the agent pick the right one. We reuse run_agent from Part 2.

In [ ]:
# A third tool with a clearly different job: return a greeting in a language.
greetings = {
    "french": "Bonjour",
    "spanish": "Hola",
    "arabic": "Marhaba",
    "swahili": "Jambo",
}

greeting_tool = {
    "name": "greet_in_language",
    "description": "Return a greeting word in a given language. Use this when the user asks how to greet someone in another language.",
    "input_schema": {
        "type": "object",
        "properties": {
            "language": {"type": "string", "description": "The language, lowercase, for example 'french'."}
        },
        "required": ["language"],
    },
}


def run_greeting(language):
    return greetings.get(language.lower().strip(), "Error: I do not know that language yet.")

In [ ]:
tools = [calculator_tool, lookup_tool, greeting_tool]

def run_tool(name, tool_input):
    if name == "calculator":
        return run_calculator(tool_input["expression"])
    if name == "lookup_fact":
        return run_lookup(tool_input["name"])
    if name == "greet_in_language":
        return run_greeting(tool_input["language"])
    return "Error: unknown tool."

SYSTEM = "You are a helpful assistant. Use the tools when they fit the request, one step at a time, then give a short final answer."

In [ ]:
print(run_agent("How do I say hello in Swahili?"))
print("\n-----\n")
print(run_agent("What is the approximate population of Bern?"))
print("\n-----\n")
print(run_agent("What is 45 plus 67? Use the calculator."))

## Part 4: Reflection and replanning

In [ ]:
# Reflection: make a draft, then judge and improve it.
def ask(prompt):
    response = client.messages.create(model=MODEL, max_tokens=1024, messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in response.content if b.type == "text")

draft = ask("Write a 2 sentence summary of why planning helps an AI agent.")
print("DRAFT:\n", draft)

critique = ask("Here is a draft:\n" + draft + "\nName one concrete way to improve it, then write the improved 2 sentence version.")
print("\nIMPROVED:\n", critique)

In [ ]:
# A weather tool that fails on its own when the city is ambiguous, so we can see
# the agent read the error and try again with a better input.
weather_data = {
    "geneva, switzerland": "18 degrees C and cloudy",
    "nairobi, kenya": "26 degrees C and sunny",
}

weather_tool = {
    "name": "get_weather",
    "description": "Get the weather for a city. Include the country, for example 'Geneva, Switzerland'.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City and country, for example 'Nairobi, Kenya'."}
        },
        "required": ["city"],
    },
}


def run_weather(city):
    key = city.lower().strip()
    if "," not in key:
        # The error message tells the model how to fix its request.
        return "Error: many cities share that name. Please include the country, like 'Geneva, Switzerland'."
    return weather_data.get(key, "Error: no data for that city. Try Geneva, Switzerland or Nairobi, Kenya.")

In [ ]:
# Replanning: the weather tool returns an error if the city has no country.
# Watch the agent read the error and try again. We reuse run_agent.
tools = [weather_tool]

def run_tool(name, tool_input):
    if name == "get_weather":
        return run_weather(tool_input["city"])
    return "Error: unknown tool."

SYSTEM = "You are a helpful assistant. Use the weather tool. If it returns an error, read the message and try again with a better input."

print(run_agent("What is the weather in Geneva?"))

## Part 5: Capstone, a field data helper

A small agent that looks up program data and does arithmetic. Reuses run_agent.

In [ ]:
# A tiny, made-up dataset for the workshop: a few program indicators by region.
# In a real project this might come from a spreadsheet or a database.
program_data = {
    "north": {"households_reached": 1200, "clinics": 4, "literacy_rate": 0.71},
    "south": {"households_reached": 900, "clinics": 3, "literacy_rate": 0.63},
    "east": {"households_reached": 1500, "clinics": 6, "literacy_rate": 0.78},
}

lookup_indicator_tool = {
    "name": "lookup_indicator",
    "description": "Look up one indicator for one region. Regions: north, south, east. Indicators: households_reached, clinics, literacy_rate.",
    "input_schema": {
        "type": "object",
        "properties": {
            "region": {"type": "string", "description": "north, south, or east"},
            "indicator": {"type": "string", "description": "households_reached, clinics, or literacy_rate"},
        },
        "required": ["region", "indicator"],
    },
}


def run_lookup_indicator(region, indicator):
    region = region.lower().strip()
    indicator = indicator.lower().strip()
    if region not in program_data:
        return "Error: unknown region. Use north, south, or east."
    if indicator not in program_data[region]:
        return "Error: unknown indicator. Use households_reached, clinics, or literacy_rate."
    return str(program_data[region][indicator])

In [ ]:
tools = [lookup_indicator_tool, calculator_tool]

def run_tool(name, tool_input):
    if name == "lookup_indicator":
        return run_lookup_indicator(tool_input["region"], tool_input["indicator"])
    if name == "calculator":
        return run_calculator(tool_input["expression"])
    return "Error: unknown tool."

SYSTEM = "You are a field data helper. Use the tools to look up indicators and to do any arithmetic, one step at a time, then give a short, clear answer."

print(run_agent("What is the total number of clinics across all three regions?"))

## Done

In one notebook you ran a tool-use loop, a ReAct agent, a multi-tool agent, reflection and replanning, and a capstone. The same loop powered all of them.